# Dummy baseline: random product predictions

This notebook randomly selects 10 distinct product IDs for each test instance. It demonstrates the submission format only; it does not match review text to product metadata and is not a BM25 or dense retrieval baseline.

Keep this notebook with `catalog.json`, `train_gold.json`, `test_queries.json`, and `category_taxonomy.txt`. Set `DATA_DIR` to that folder if needed, replace `studentID` in the output filename with your student ID, and run the code cell. Only Python's standard library is required.

The random baseline reads only `catalog.json` and `test_queries.json`. Training labels and the 500 category paths (one per line in `category_taxonomy.txt`) are available for developing your own methods but are not used here.

The CSV contains `query_id,item_id_1,...,item_id_10`, with one row per test instance in the original order. For your own method, put the highest-ranked product first. No category predictions or scores are submitted. Running the cell again with the same seed and input order reproduces the same predictions and overwrites the output CSV.

In [ ]:
import csv
import json
import random
from pathlib import Path

SEED = 42
DATA_DIR = Path(".")
SUBMISSION_PATH = Path("studentID_final.csv")  # Replace studentID with your ID.
TOP_K = 10

rng = random.Random(SEED)

def load_json(path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

catalog = load_json(DATA_DIR / "catalog.json")
test_queries = load_json(DATA_DIR / "test_queries.json")

catalog_item_ids = [item["item_id"] for item in catalog]
catalog_id_set = set(catalog_item_ids)
test_instance_ids = [instance["query_id"] for instance in test_queries]

assert len(catalog_item_ids) == 50_000
assert len(set(catalog_item_ids)) == len(catalog_item_ids)
assert len(test_queries) == 19_858
assert len(set(test_instance_ids)) == len(test_instance_ids)
assert all(len(instance["reviews"]) == 5 for instance in test_queries)

fieldnames = ["query_id"] + [
    f"item_id_{rank}" for rank in range(1, TOP_K + 1)
]

with SUBMISSION_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

    for instance in test_queries:
        ranked_items = rng.sample(catalog_item_ids, TOP_K)
        row = {"query_id": instance["query_id"]}
        row.update({
            f"item_id_{rank}": item_id
            for rank, item_id in enumerate(ranked_items, start=1)
        })
        writer.writerow(row)

with SUBMISSION_PATH.open("r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    generated_rows = list(reader)

assert reader.fieldnames == fieldnames
assert len(generated_rows) == len(test_queries)
assert [row["query_id"] for row in generated_rows] == test_instance_ids
for row in generated_rows:
    assert set(row) == set(fieldnames)
    predicted_ids = [row[f"item_id_{rank}"] for rank in range(1, TOP_K + 1)]
    assert len(set(predicted_ids)) == TOP_K
    assert all(item_id in catalog_id_set for item_id in predicted_ids)

print(f"Saved {len(generated_rows):,} predictions to {SUBMISSION_PATH}")
print("Columns:", fieldnames)
